In [1]:
from antares_devkit.models import BaseFilter
from antares_devkit.models import DevKitLocus
from antares_client import search
import numpy as np
import matplotlib.pyplot as plt
from astropy.time import Time
import pandas as pd
import math
from astropy.coordinates import Angle, SkyCoord
import astropy.units as u
import time
import warnings
import itertools

In [7]:
#extract data from lsst alert
def lsst_alert(alert):
    obs_date = float(alert.properties['lsst_diaSource_midpointMjdTai'])
    oid_mag = float(-2.5*np.log10(alert.properties['lsst_diaSource_scienceFlux']) + 31.4)
    temp_mag = float(-2.5*np.log10(alert.properties['lsst_diaSource_templateFlux']) + 31.4)
    diff_mag = float(temp_mag - oid_mag)
    
    bandpass = alert.properties['lsst_diaSource_band']
    band = 'lsst_' + str(bandpass)

    if alert.properties['lsst_diaSource_isNegative'] == True:
        pos_or_neg = -1
    else:
        pos_or_neg = 1
        
    return {band: [obs_date, oid_mag, temp_mag, diff_mag, pos_or_neg]}

def lsst_quality(alert):
    quality = True
    quality &= alert.properties['lsst_diaSource_reliability'] >= 0.6
    quality &= alert.properties['lsst_diaSource_psfFlux_flag'] == False
    quality &= alert.properties['lsst_diaSource_centroid_flag'] == False
    quality &= alert.properties['lsst_diaSource_shape_flag'] == False
    quality &= alert.properties['lsst_diaSource_isDipole'] == False
    quality &= alert.properties['lsst_diaSource_pixelFlags_saturated'] == False
    quality &= alert.properties['lsst_diaSource_pixelFlags_edge'] == False
    quality &= alert.properties['lsst_diaSource_pixelFlags_cr'] == False
    quality &= alert.properties['lsst_diaSource_pixelFlags_streak'] == False
    quality &= alert.properties['lsst_diaSource_scienceFlux'] >= 0
    quality &= alert.properties['lsst_diaSource_templateFlux'] >= 0

    if 'lsst_diaSource_ssObjectId' in alert.properties:
        quality &= alert.properties['lsst_diaSource_ssObjectId'] in (0, None, "0")
    return quality

#extract data from ztf alert
#uses formula from 260609 6/15 markdown cell
def ztf_alert(alert):
    bands = ['x', 'g', 'R', 'i']
    
    band_id = alert.properties['ztf_fid']
    obs_date = float(alert.properties['ant_mjd'])
    ant_mag = alert.properties['ant_mag']
    temp_mag = alert.properties['ztf_magnr']
    diff_img_mag = alert.properties['ztf_magpsf']
    
    if alert.properties['ztf_isdiffpos'] == 'f' or alert.properties['ztf_isdiffpos'] == 0:
        diff_mag = -2.5* np.log10(1 - 10.**(0.4*(temp_mag - diff_img_mag)))
        pos_or_neg = -1
    else:
        diff_mag = -2.5* np.log10(10.**(0.4*(temp_mag - diff_img_mag)) + 1)
        pos_or_neg = 1

    oid_mag = diff_mag + temp_mag
    band = 'ztf_' + str(bands[band_id])
    return {band: [obs_date, oid_mag, temp_mag, diff_mag, pos_or_neg]}

#checks ztf alert for good quality
def ztf_quality(alert):
    quality = True
    quality &= alert.properties['ztf_rb'] >= 0.65
    quality &= alert.properties['ztf_nbad'] == 0
    quality &= alert.properties['ztf_fwhm'] <= 5
    quality &= alert.properties['ztf_elong'] <= 1.2
    quality &= abs(alert.properties['ztf_magdiff']) <= 0.1
    return quality

#unpacks all alerts (that pass a quality test)
def oid_get_info(alerts, latest_mjd):
    oid = {'lsst_g': 0, 'lsst_r': 0, 'lsst_i': 0, 'lsst_z': 0, 'lsst_u': 0, 'lsst_y': 0, 
               'ztf_g': 0, 'ztf_R': 0, 'ztf_i': 0}

    #snr check
    if alerts[-1].properties['ant_survey'] == 4:
        if alerts[-1].properties['lsst_diaSource_snr'] < 8:
            return 'snr too low'
        elif lsst_quality(alerts[-1]) == False:
            return 'lsst alert quality fail'
        else:  # check if bright
            trig_alert = lsst_alert(alerts[-1])
            band = [*trig_alert][0]
            if trig_alert[band][1] > 21:
                return 'triggered alert too dim'
    elif alerts[-1].properties['ant_survey'] == 1:
        if (1.0857/alerts[-1].properties['ztf_sigmapsf']) < 8:
            return 'snr too low'

    for alert in alerts:
        #check mjd
        if alert.properties['ant_mjd'] < (math.floor(latest_mjd) - 30):
            continue

        # quality check
        if alert.properties['ant_survey'] == 4:
            lsst_pass = lsst_quality(alert)
            if lsst_pass == True:
                comp = lsst_alert(alert)
            else:
                continue
        elif alert.properties['ant_survey'] == 1:
            ztf_pass = ztf_quality(alert)
            if ztf_pass == True:
                comp = ztf_alert(alert)
            else:
                continue
        else:
            continue
            
        band = [*comp][0]
        #put quality data together organized by band
        if isinstance(oid[band], pd.DataFrame) == 0:
            oid_df = pd.DataFrame([comp[band]], columns = ['mjd', 'oid_mag', 'template_mag', 'diff_mag', 'pos/neg'])
            oid[band] = oid_df
            continue
        else:
            data_df = pd.DataFrame([comp[band]], columns = ['mjd', 'oid_mag', 'template_mag', 'diff_mag', 'pos/neg'])
            current_data = oid[band]
            oid[band] = pd.concat([current_data, data_df]).drop_duplicates().reset_index(drop=True)

    #checking if any data passed
    fail = 0
    for band in oid:
        if isinstance(oid[band], pd.DataFrame) == False:
            fail += 1
    if fail == 9:
        return 'no recent data'
    else:
        return oid

def var_params(oid):
    vp = {'lsst_g': 0, 'lsst_r': 0, 'lsst_i': 0, 'lsst_z': 0, 'lsst_u': 0, 'lsst_y': 0, 
               'ztf_g': 0, 'ztf_R': 0, 'ztf_i': 0}
    data_lim = 0
    for band in oid:
        if band == 'locus':
            continue
        if isinstance(oid[band], pd.DataFrame) == 0:
            data_lim += 1
            continue
        if len(oid[band]) < 3:
            data_lim += 1
            vp[band] = {'diff_mag':0, 'var':0, 'hyper_v':0, 'data':oid[band]}
        
        max_mjd_i = oid[band].mjd.idxmax()
        diff_mag = float(oid[band]['diff_mag'][max_mjd_i])
        var = np.var(oid[band]['oid_mag'])
        # hv = float(np.sqrt(diff_mag ** 2 + (coeff * var) ** 2))
        # vp[band] = {'diff_mag':diff_mag, 'var':var, 'hyper_v':hv, 'data':oid[band]}
        vp[band] = {'diff_mag':diff_mag, 'var':var, 'data':oid[band]}
        
    if data_lim == 9:
        return 0
    else: 
        return vp

def catalog_check(locus):
    flags = {'known_star': False, 'known_agn': False, 'known_galaxy': False, 'gaia_likely_star': False, 
                            'gaia_astro_star': False, 'ps1_pointlike': False}
    for cat in locus.catalog_objects:
        if cat == 'asassn_variable_catalog' or cat == 'asassn_variable_Catalog_v2_20190802' or cat == 'sdss_stars':
            flags['known_star'] = True
        if cat == 'veron_agn_qso' or cat == 'milliquas':
            flags['known_agn'] = True
        if cat == '2mass_xsc' or cat == 'nyu_valueadded_gals' or cat == 'sdss_gals' or cat == 'RC3':
            flags['known_galaxy'] = True
        if cat == 'gaia_dr3_gaia_source':
            if isinstance(locus.catalog_objects[cat][0]['classprob_dsc_combmod_star'], float) == 1 and locus.catalog_objects[cat][0]['classprob_dsc_combmod_star'] > 0.5:
                flags['gaia_likely_star'] = True
            if isinstance(locus.catalog_objects[cat][0]['parallax'], float) == 1:
                gdr3_check1 = (locus.catalog_objects[cat][0]['parallax'] + 0.017)/locus.catalog_objects[cat][0]['parallax_error']
                alpha = locus.catalog_objects[cat][0]['pmra']/locus.catalog_objects[cat][0]['pmra_error']
                beta = locus.catalog_objects[cat][0]['pmdec']/locus.catalog_objects[cat][0]['pmdec_error']
                gamma = locus.catalog_objects[cat][0]['pmra_pmdec_corr']
                gdr3_check2 = np.sqrt((alpha**2 + beta**2 -2*alpha*beta*gamma)/(1 - gamma**2))
                if gdr3_check1 > 5 or gdr3_check2 > 5:
                    flags['gaia_astro_star'] = True
        alert_temp = locus.alerts[-1].properties
        if alert_temp['ant_survey'] == 1 and ((alert_temp['ztf_sgscore1'] > 0.5 and alert_temp['ztf_distpsnr1'] < 1) or (alert_temp['ztf_sgscore2'] > 0.5 and alert_temp['ztf_distpsnr2'] < 1) or (alert_temp['ztf_sgscore3'] > 0.5 and alert_temp['ztf_distpsnr3'] < 1)):
            flags['ps1_pointlike'] = True
    return flags

def query_by_chunk(mjd_start, mjd_end, ra_min, ra_max, dec_min, dec_max,
                   faint_lim=FAINT_MAG_LIM, min_ndet=MIN_NDET, max_results=ANTARES_MAX_RETURN,
                   retries=3, wait=15):
    # set the query to fetch loci with alerts matching search criteria
    query = {
        "query": {
            "bool": {
                "filter": [
                    { "range": { "properties.newest_alert_observation_time": { "gte": mjd_start, "lte": mjd_end} } },
                    { "range": { "properties.brightest_alert_magnitude": {"lte": faint_lim} } },
                    { "range": { "ra": { "gte": ra_min, "lte": ra_max} } },
                    { "range": { "dec": { "gte": dec_min, "lte": dec_max} } },
                    { "range": { "properties.num_alerts": { "gte": min_ndet} } },
                    { "exists": { "field": "properties.survey.lsst" } }
                      ]
            }
        }
    }

    # sometimes the ANTARES server times out so try a few times
    # before giving up and returning an empty list (to avoid crashing the entire query)
    for attempt in range(retries):
        try:
            loci = list(search.search(query))
            if len(loci) >= max_results:
                warnings.warn(
                    f'Chunk MJD {mjd_start:.1f}–{mjd_end:.1f} hit the {max_results}-result max.'
                    f'— Reduce QUERY_DT or this window is incomplete.',
                    stacklevel=2
                )
            return get_ids(loci)
        except Exception as e:
            if attempt < retries - 1:
                print(f'  Chunk {mjd_start}–{mjd_end} failed (attempt {attempt+1}/{retries}): {e}.')
                time.sleep(wait)
            else:
                print(f'  Chunk {mjd_start}–{mjd_end} giving up after {retries} attempts.')
                return []

def get_ids(loci):
    # print(loci)
    loci_ids = []
    for locus in loci:
        loci_ids.append(locus.locus_id)
    return loci_ids

def get_target_info(locus, oid, make_csv=False):
    for cat in locus.catalog_objects:
        sdss_id = None
        gaia_id = None
        gaia_bp = None
        gaia_rp = None
        gaia_g = None
        twomass_id = None
        
        if cat == 'sdss_gals':
            sdss_id = locus.catalog_objects[cat][0]['Objid']
        elif cat == 'sdss_stars':
            sdss_id = locus.catalog_objects[cat][0]['Objid']
        elif cat == 'gaia_dr3_gaia_source':
            gaia_id = locus.catalog_objects[cat][0]['Source Id']
            gaia_bp = locus.catalog_objects[cat][0]['Phot Bp Mean Mag']
            gaia_rp = gaia_bp - locus.catalog_objects[cat][0]['Bp Rp']
            gaia_g = gaia_bp - locus.catalog_objects[cat][0]['Bp G']
        elif cat == '2mass_xsc':
            twomass_id = locus.catalog_objects[cat][0]['Id']

    sky_brightness_mode = 'dark'
    ra = locus.ra
    dec = locus.dec
    for band in oid:
        u = None
        g = None
        r = None 
        i = None
        z = None
        g_mjd = 0
        r_mjd = 0
        i_mjd = 0
        if band == 'lsst_u':
            u = oid[band]['oid_mag'][-1]
        elif band == 'lsst_z':
            z = oid[band]['oid_mag'][-1]
        elif band == 'lsst_g' or band == 'ztf_g':
            if oid[band]['mjd'][-1] > g_mjd:
                g_mjd = oid[band]['mjd'][-1]
                g = oid[band]['oid_mag'][-1]
        elif band == 'lsst_r' or band == 'ztf_R':
            if oid[band]['mjd'][-1] > r_mjd:
                r_mjd = oid[band]['mjd'][-1]
                r = oid[band]['oid_mag'][-1]
        elif band == 'lsst_i' or band == 'ztf_i':
            if oid[band]['mjd'][-1] > i_mjd: 
                i_mjd = oid[band]['mjd'][-1]
                i = oid[band]['oid_mag'][-1]
    can_offset = True
    n_exposures = 1
    target = [None, 'hypervariable', 'APOGEE', None, sdss_id, gaia_id, twomass_id, 'dark', 
              ra, dec, None, None, None, None, lambda eff!!!!!!!!!!, u, g, r, i, z, optical prov!!!!!!!!!, 
            gaia_bp, gaia_rp, gaia_g, None, delta_ra!!!!!!!!, delta_dec!!!!!!, True, True, 1, True, 
            t_now, t_now+14, False, added_on!!!!!!!!!]
    
    if make_csv == False: 
        return #dict
    else:
        #make csv file and return it
        a = 1
        
def hyperv_pass(oid, coeff=COEFF):
    for band in oid:
        if band == 'locus':
            continue
        if isinstance(oid[band], pd.DataFrame) == 0:
            continue
        if len(oid[band]) < 3:
            continue
        
        max_mjd_i = oid[band].mjd.idxmax()
        diff_mag = float(oid[band]['diff_mag'][max_mjd_i])
        var = np.var(oid[band]['oid_mag'])
        hv = float(np.sqrt(diff_mag ** 2 + (coeff * var) ** 2))

        if hv >= hv_thresh
            return 1
    return 0

SyntaxError: invalid syntax (1112651397.py, line 280)

In [2]:
snr_f = 0
recent_f = 0
lsst_q_f = 0
mag_f = 0
vp_dl_f = 0
variable_f = 0
checked = 0

oid_sample = {}
ra_s = np.arange(0, 360, 10)
dec_s = np.arange(-90, 40, 10)
# ra_s = [0.0]
# dec_s = [0.0]

FAINT_MAG_LIM = 21.0
MIN_NDET = 3
ANTARES_MAX_RETURN = 10000
DAY_LIM = 30
t_now = Time.now().mjd

for ra, dec in list(itertools.product(ra_s, dec_s)):
    RA_MIN = float(ra)
    RA_MAX = float(RA_MIN + 10)
    DEC_MIN = float(dec)
    DEC_MAX = float(DEC_MIN + 10)
    
    loci_ids = query_by_chunk(t_now-100, t_now, RA_MIN, RA_MAX, DEC_MIN, DEC_MAX)
    print(f'({RA_MIN} - {RA_MAX}, {DEC_MIN} - {DEC_MAX}) : {len(loci_ids)}')
    
    for l_id in loci_ids:
        locus = search.get_by_id(l_id)
        oid_dv = locus.to_devkit()
        oid_locus = DevKitLocus.model_validate(oid_dv)
        
        oid_here = oid_get_info(oid_locus.alerts, oid_locus.properties['newest_alert_observation_time'])
    
        if isinstance(oid_here, str):
            if oid_here == 'snr too low':
                snr_f += 1
            elif oid_here == 'no recent data':
                recent_f += 1
            elif oid_here == 'lsst alert quality fail':
                lsst_q_f += 1
            elif oid_here == 'triggered alert too dim':
                mag_f += 1
        else:
            hv = hv_pass(oid_here, 6, 1)
            if hv:
                


_IncompleteInputError: incomplete input (1362855264.py, line 49)

In [4]:
t_now = Time.now().mjd
t_now

NameError: name 'Time' is not defined